In [20]:
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Suppress warning messages during pipeline execution
warnings.filterwarnings('ignore')

print('====================================================================')
print('=== COMPONENT 1: SALES & FINANCIAL CREDIT READINESS ANALYTICS ===')
print('====================================================================\n')

=== COMPONENT 1: SALES & FINANCIAL CREDIT READINESS ANALYTICS ===



In [21]:
# =========================================================================
# STEP 1: DATA LOADING & PRELIMINARY CLEANING
# =========================================================================
# Load financial and sales dataset for merchant credit scoring
try:
  df = pd.read_csv('sales & financial.csv')
except FileNotFoundError:
  # Creating synthetic dataset if actual file is missing
  np.random.seed(42)
  n = 1000
  df = pd.DataFrame({
      'monthly_revenue_rs': np.random.uniform(100000, 2000000, n),
      'monthly_expenses_rs': np.random.uniform(50000, 1500000, n),
      'digital_payment_ratio': np.random.uniform(0.1, 0.9, n),
      'months_active': np.random.randint(1, 60, n),
      'stockout_rate': np.random.uniform(0.0, 0.3, n),
      'profit_margin_pct': np.random.uniform(5, 35, n),
      'target_credit_ready': np.random.choice([0, 1], size=n, p=[0.4, 0.6]),
  })

# Remove duplicate records to prevent data leakage
df = df.drop_duplicates().reset_index(drop=True)

# Safe drop of identifier columns
df = df.drop(columns=['shop_id', 'year_month'], errors='ignore')


In [22]:
# =========================================================================
# STEP 2: DOMAIN FINANCIAL FEATURE ENGINEERING
# =========================================================================

# --- Feature A: Net Cash Flow ---
if 'monthly_expenses_rs' in df.columns and 'monthly_revenue_rs' in df.columns:
  df['net_cash_flow'] = df['monthly_revenue_rs'] - df['monthly_expenses_rs']

# --- Feature B: Debt-to-Income (DTI) Ratio ---
if 'monthly_expenses_rs' in df.columns and 'monthly_revenue_rs' in df.columns:
  df['debt_to_income_ratio'] = df['monthly_expenses_rs'] / (
      df['monthly_revenue_rs'] + 1e-5
  )

# --- Feature C: Digital Revenue Volume ---
if (
    'monthly_revenue_rs' in df.columns
    and 'digital_payment_ratio' in df.columns
):
  df['digital_revenue_volume'] = (
      df['monthly_revenue_rs'] * df['digital_payment_ratio']
  )

# --- Feature D: Revenue per Active Month ---
if 'months_active' in df.columns and 'monthly_revenue_rs' in df.columns:
  df['revenue_per_active_month'] = df['monthly_revenue_rs'] / (
      df['months_active'] + 1e-5
  )

# --- Feature E: Cash Flow Margin ---
if 'net_cash_flow' in df.columns and 'monthly_revenue_rs' in df.columns:
  df['cash_flow_margin'] = df['net_cash_flow'] / (
      df['monthly_revenue_rs'] + 1e-5
  )

# --- UPGRADE 1 FEATURE: Safe Loan Capacity Target (For Regression) ---
if 'net_cash_flow' in df.columns:
  # Safe loan target calculated based on Net Cash Flow & Current Credit Readiness Status
  df['recommended_loan_limit'] = np.maximum(
      0, df['net_cash_flow'] * 3.5 * df['target_credit_ready']
  )


In [23]:
# =========================================================================
# STEP 3: TARGET & FEATURE SEPARATION + PREPROCESSING PIPELINE
# =========================================================================
# Separation of Features and Targets
X = df.drop(
    columns=['target_credit_ready', 'recommended_loan_limit'], errors='ignore'
)
y_cls = df['target_credit_ready'].astype(int)
y_reg = df['recommended_loan_limit'].astype(float)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imp', SimpleImputer(strategy='median')),
                ('sc', StandardScaler()),
            ]),
            num_cols,
        ),
        (
            'cat',
            Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore')),
            ]),
            cat_cols,
        ),
    ]
)

# Stratified Split for Classification, parallel split for Regression target
(
    X_train,
    X_test,
    y_train_cls,
    y_test_cls,
    y_train_reg,
    y_test_reg,
) = train_test_split(
    X, y_cls, y_reg, test_size=0.20, random_state=42, stratify=y_cls
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [29]:
# =========================================================================
# STEP 4: MODEL PIPELINES (CLASSIFICATION & REGRESSION)
# =========================================================================
# Classifier Pipelines
models = {
    'LogisticRegression': Pipeline([
        ('prep', preprocessor),
        (
            'clf',
            LogisticRegression(
                max_iter=1000, class_weight='balanced', random_state=42
            ),
        ),
    ]),
    'DecisionTree': Pipeline([
        ('prep', preprocessor),
        (
            'clf',
            DecisionTreeClassifier(
                max_depth=6,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=42,
            ),
        ),
    ]),
    'RandomForest': Pipeline([
        ('prep', preprocessor),
        (
            'clf',
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                class_weight='balanced',
                n_jobs=1,
                random_state=42,
            ),
        ),
    ]),
    'GradientBoosting': Pipeline([
        ('prep', preprocessor),
        (
            'clf',
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.85,
                random_state=42,
            ),
        ),
    ]),
}

try:
  from xgboost import XGBClassifier

  models['XGBoost'] = Pipeline([
      ('prep', preprocessor),
      (
          'clf',
          XGBClassifier(
              n_estimators=300,
              max_depth=4,
              learning_rate=0.05,
              subsample=0.85,
              colsample_bytree=0.85,
              eval_metric='logloss',
              random_state=42,
          ),
      ),
  ])
except ImportError:
  print('Notice: XGBoost not installed. Running with Sklearn Models.')


In [30]:
# =========================================================================
# STEP 5: CROSS-VALIDATION BENCHMARKING & CHAMPION SELECTION
# =========================================================================
print('=== Running 5-Fold Cross Validation Benchmarking ===')
cv_results = {}

for name, m in models.items():
  scores = cross_validate(
      m,
      X_train,
      y_train,
      cv=cv,
      scoring=['roc_auc', 'accuracy', 'f1'],
      n_jobs=-1,
  )
  cv_results[name] = {
      'auc': scores['test_roc_auc'].mean(),
      'acc': scores['test_accuracy'].mean(),
      'f1': scores['test_f1'].mean(),
  }
  print(
      f'  {name:20s} | CV ROC-AUC: {cv_results[name]["auc"]:.3f} | CV Acc:'
      f' {cv_results[name]["acc"]:.3f} | CV F1: {cv_results[name]["f1"]:.3f}'
  )

# Select Champion Model based on highest ROC-AUC score
best_model_name = max(cv_results, key=lambda k: cv_results[k]['auc'])
print(f'\n🏆 Champion Model Selected: {best_model_name}')

champion_model = models[best_model_name]
champion_model.fit(X_train, y_train)

# Model Predictions on Test Dataset
y_pred = champion_model.predict(X_test)
y_proba = champion_model.predict_proba(X_test)[:, 1]

print('\n===================================================')
print(f'=== Champion Model ({best_model_name}) Test Performance ===')
print('===================================================')
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision : {precision_score(y_test, y_pred):.3f}')
print(f'Recall    : {recall_score(y_test, y_pred):.3f}')
print(f'F1-Score  : {f1_score(y_test, y_pred):.3f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_proba):.3f}\n')
print(
    classification_report(
        y_test, y_pred, target_names=['Not Credit Ready', 'Credit Ready']
    )
)


=== Running 5-Fold Cross Validation Benchmarking ===
  LogisticRegression   | CV ROC-AUC: 0.883 | CV Acc: 0.806 | CV F1: 0.808
  DecisionTree         | CV ROC-AUC: 0.790 | CV Acc: 0.724 | CV F1: 0.726
  RandomForest         | CV ROC-AUC: 0.857 | CV Acc: 0.777 | CV F1: 0.779
  GradientBoosting     | CV ROC-AUC: 0.860 | CV Acc: 0.780 | CV F1: 0.782
  XGBoost              | CV ROC-AUC: 0.857 | CV Acc: 0.776 | CV F1: 0.778

🏆 Champion Model Selected: LogisticRegression

=== Champion Model (LogisticRegression) Test Performance ===
Accuracy  : 0.833
Precision : 0.856
Recall    : 0.801
F1-Score  : 0.828
ROC-AUC   : 0.906

                  precision    recall  f1-score   support

Not Credit Ready       0.81      0.86      0.84       199
    Credit Ready       0.86      0.80      0.83       201

        accuracy                           0.83       400
       macro avg       0.83      0.83      0.83       400
    weighted avg       0.83      0.83      0.83       400



In [31]:
# =========================================================================
# STEP 6: UPGRADE 1 — REGRESSION MODEL TRAINING (CREDIT LIMIT PREDICTION)
# =========================================================================
print('\n=== Training Credit Limit Prediction Engine (Regression) ===')

credit_limit_regressor = Pipeline([
    ('prep', preprocessor),
    (
        'reg',
        RandomForestRegressor(
            n_estimators=200,
            max_depth=8,
            random_state=42,
            n_jobs=1,  # Fixed: Resolves Python 3.14 Jupyter Event Loop Conflict
        ),
    ),
])

# Train Regressor model using qualified credit-ready training data
credit_limit_regressor.fit(
    X_train[y_train_cls == 1], y_train_reg[y_train_cls == 1]
)

# Predict on test set
y_reg_pred = credit_limit_regressor.predict(X_test)

print(
    'Regression Model MAE : LKR'
    f' {mean_absolute_error(y_test_reg[y_test_cls == 1], y_reg_pred[y_test_cls == 1]):,.2f}'
)
print(
    'Regression Model R2  :'
    f' {r2_score(y_test_reg[y_test_cls == 1], y_reg_pred[y_test_cls == 1]):.3f}'
)


=== Training Credit Limit Prediction Engine (Regression) ===
Regression Model MAE : LKR 143.18
Regression Model R2  : 1.000


In [32]:
# =========================================================================
# STEP 7: UPGRADE 4 — HYBRID ML + RULE ENGINE INTEGRATION
# =========================================================================
print('\n===================================================')
print('=== HYBRID ML & RULE ENGINE EVALUATION SIMULATION ===')
print('===================================================')


def evaluate_merchant_credit_decision(merchant_row, cls_model, reg_model):
  """Evaluates credit eligibility by combining ML scoring, Credit Limit prediction,

  and Hard Business Rule Engine guardrails.
  """
  raw_data = merchant_row.iloc[0]

  # ML Classification & Regression Output
  prob_score = round(cls_model.predict_proba(merchant_row)[0, 1] * 100)
  predicted_limit = max(0, round(reg_model.predict(merchant_row)[0], -3))

  # --- HARD BUSINESS RULES GUARDRAILS ---
  hard_blocks = []

  # Rule 1: Debt-To-Income Ratio Guardrail (> 85% is critical)
  if raw_data.get('debt_to_income_ratio', 0) > 0.85:
    hard_blocks.append('CRITICAL: DTI Ratio exceeds safety threshold (>85%).')

  # Rule 2: Active Months Cut-off (< 3 months operating history)
  if raw_data.get('months_active', 100) < 3:
    hard_blocks.append(
        'HIGH RISK: Business operating history is less than 3 months.'
    )

  # Rule 3: High Stockout Rate Rule (> 25% stockouts)
  if raw_data.get('stockout_rate', 0) > 0.25:
    hard_blocks.append('OPERATIONAL RISK: Stockout rate exceeds 25%.')

  # --- FINAL HYBRID DECISION ---
  if hard_blocks:
    final_status = '🔴 REJECTED BY RULE ENGINE (Hard Block)'
    recommended_limit = 0
  elif prob_score >= 70:
    final_status = '🟢 APPROVED (Tier 1 Prime Merchant)'
    recommended_limit = predicted_limit
  elif prob_score >= 50:
    final_status = '🟡 CONDITIONAL APPROVAL (Tier 2 Micro-Loan)'
    recommended_limit = min(predicted_limit, 250000)  # Capped limit
  else:
    final_status = '🔴 REJECTED (High Risk ML Score)'
    recommended_limit = 0

  return {
      'credit_score': f'{prob_score}/100',
      'final_status': final_status,
      'recommended_max_loan_lkr': f'LKR {recommended_limit:,.2f}',
      'rule_engine_alerts': hard_blocks if hard_blocks else ['None'],
  }


# Test evaluation on a sample test merchant
sample_merchant = X_test.iloc[[0]]
evaluation_result = evaluate_merchant_credit_decision(
    sample_merchant, champion_cls_model, credit_limit_regressor
)

print(f'Merchant Reference Index : {X_test.index[0]}')
print(f'Credit Readiness Score   : {evaluation_result["credit_score"]}')
print(f'Final Credit Status      : {evaluation_result["final_status"]}')
print(
    'Max Safe Loan Amount     :'
    f' {evaluation_result["recommended_max_loan_lkr"]}'
)
print(f'Rule Engine Alerts       : {evaluation_result["rule_engine_alerts"]}')



=== HYBRID ML & RULE ENGINE EVALUATION SIMULATION ===
Merchant Reference Index : 438
Credit Readiness Score   : 22/100
Final Credit Status      : 🔴 REJECTED (High Risk ML Score)
Max Safe Loan Amount     : LKR 0.00
Rule Engine Alerts       : ['None']


In [33]:
# =========================================================================
# STEP 8: MODEL SERIALIZATION
# =========================================================================
bundle = {
    'classifier_model': champion_cls_model,
    'regressor_model': credit_limit_regressor,
    'feature_names': X.columns.tolist(),
    'numerical_cols': num_cols,
    'categorical_cols': cat_cols,
}

model_filename = 'component1_sales_financial_model.pkl'
joblib.dump(bundle, model_filename)
print(f'\n✅ Complete Upgraded Component 1 Bundle saved as "{model_filename}"')


✅ Complete Upgraded Component 1 Bundle saved as "component1_sales_financial_model.pkl"
